### RAG end to end pipeline ###

In [21]:
import tqdm
import os
from pathlib import Path
from langchain_community import document_loaders
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [26]:
# Function to process PDF documents and extract text
def process_documents(doc_dir):
    doc_dir = "../data"
    all_documents = []
    pdf_directory = Path(doc_dir)
    pdf_files = list(pdf_directory.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf in pdf_files:
        print(f"Processing {pdf}...")

        try:
            loader = PyPDFLoader(str(pdf))
            documents = loader.load()

            ##adding the metadata to the documents

            for doc in documents:
                doc.metadata["source"] = pdf.name

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf}: {e}")   

    print(f"Total documents processed: {len(all_documents)}")
    return all_documents  
path="../data"
all_documents = process_documents(path)


Found 2 PDF files in ..\data
Processing ..\data\attention.pdf...
Processing ..\data\research_paper.pdf...
Total documents processed: 42


In [30]:
### text splitting the documents into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    all_chunks = []
    for doc in documents:
        chunks = text_splitter.split_documents([doc])
        all_chunks.extend(chunks)

    print(f"Total chunks created: {len(all_chunks)}")
    return all_chunks

split_documents(all_documents)

Total chunks created: 168


[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 